In [ ]:
# mmcv has no Python 3.12 wheel and fails to build from source, so we skip it
# entirely and stub it out below. Only mmengine + mmsegmentation are needed.
!pip install -q mmengine
!pip install -q mmsegmentation
!pip install -q timm scipy

In [ ]:
import os, sys

if not os.path.exists('/content/SegVit'):
    !git clone https://github.com/zbwxp/SegVit.git /content/SegVit

%cd /content/SegVit
sys.path.insert(0, '/content/SegVit')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Compatibility shims
# ─────────────────────────────────────────────────────────────────────────────
import sys, types, logging
import numpy as np
import torch
import torch.nn as nn
from PIL import Image as _PILImage

# ── 1. Stub out the entire mmcv package ──────────────────────────────────────
# mmcv has no Python 3.12 wheel; we only need a handful of things from it.

class _ConvModule(nn.Module):
    """Minimal drop-in for mmcv.cnn.ConvModule (Conv + optional BN + optional activation)."""
    def __init__(self, in_channels, out_channels, kernel_size, stride=1,
                 padding=0, dilation=1, groups=1, bias='auto',
                 conv_cfg=None, norm_cfg={'type': 'BN'}, act_cfg={'type': 'ReLU'},
                 inplace=True, padding_mode='zeros', order=('conv','norm','act'), **kw):
        super().__init__()
        has_norm = norm_cfg is not None
        has_act  = act_cfg  is not None
        if bias == 'auto':
            bias = not has_norm
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size,
                              stride=stride, padding=padding, dilation=dilation,
                              groups=groups, bias=bias, padding_mode=padding_mode)
        self.norm = nn.BatchNorm2d(out_channels) if has_norm else None
        if has_act:
            _t = act_cfg.get('type', 'ReLU')
            self.activate = ({'ReLU': nn.ReLU(inplace), 'GELU': nn.GELU(),
                              'SiLU': nn.SiLU(), 'Sigmoid': nn.Sigmoid()}
                             .get(_t, nn.ReLU(inplace)))
        else:
            self.activate = None

    def forward(self, x):
        x = self.conv(x)
        if self.norm     is not None: x = self.norm(x)
        if self.activate is not None: x = self.activate(x)
        return x

def _imread(filename, flag='color', channel_order='bgr', backend=None):
    arr = np.array(_PILImage.open(filename).convert('RGB'))
    return arr[:, :, ::-1].copy() if channel_order == 'bgr' else arr

# trunc_normal_ — try timm first, fall back to torch
try:
    from timm.models.layers import trunc_normal_ as _trunc_normal_
except ImportError:
    try:
        from timm.layers import trunc_normal_ as _trunc_normal_
    except ImportError:
        from torch.nn.init import trunc_normal_
        _trunc_normal_ = trunc_normal_

import mmengine.runner as _eng_runner

# Build the fake mmcv module tree
_mmcv          = types.ModuleType('mmcv')
_mmcv_cnn      = types.ModuleType('mmcv.cnn')
_mmcv_cnn_util = types.ModuleType('mmcv.cnn.utils')
_mmcv_runner   = types.ModuleType('mmcv.runner')
_mmcv_utils    = types.ModuleType('mmcv.utils')
_mmcv_image    = types.ModuleType('mmcv.image')

_mmcv_cnn.ConvModule          = _ConvModule
_mmcv_cnn.trunc_normal_       = _trunc_normal_
_mmcv_cnn_util.trunc_normal_  = _trunc_normal_
_mmcv_cnn.utils               = _mmcv_cnn_util
_mmcv_runner.load_checkpoint  = _eng_runner.load_checkpoint
_mmcv_utils.print_log         = lambda *a, **k: None
_mmcv_image.imread            = _imread
_mmcv.imread                  = _imread
_mmcv.cnn     = _mmcv_cnn
_mmcv.runner  = _mmcv_runner
_mmcv.utils   = _mmcv_utils
_mmcv.image   = _mmcv_image

for _name, _mod in [
    ('mmcv', _mmcv), ('mmcv.cnn', _mmcv_cnn), ('mmcv.cnn.utils', _mmcv_cnn_util),
    ('mmcv.runner', _mmcv_runner), ('mmcv.utils', _mmcv_utils), ('mmcv.image', _mmcv_image),
]:
    sys.modules[_name] = _mod

# ── 2. Patch mmseg.models.builder for mmseg 1.x ─────────────────────────────
import mmseg.models.builder as _builder
from mmseg.registry import MODELS
for _n in ('BACKBONES', 'HEADS', 'SEGMENTORS', 'LOSSES'):
    if not hasattr(_builder, _n):
        setattr(_builder, _n, MODELS)

# ── 3. Stub mmseg.utils.get_root_logger (removed in mmseg 1.x) ───────────────
import mmseg.utils as _mmseg_utils
if not hasattr(_mmseg_utils, 'get_root_logger'):
    _mmseg_utils.get_root_logger = lambda log_level='INFO': logging.getLogger('mmseg')

# ── 4. Import SegViT modules to fire @register_module() decorators ────────────
import backbone
import decode_heads
import losses

import mmseg
print(f'PyTorch:        {torch.__version__}')
print(f'MMSeg:          {mmseg.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

In [ ]:
from huggingface_hub import hf_hub_download

os.makedirs('/content/checkpoints', exist_ok=True)
checkpoint_path = '/content/checkpoints/ade_51.3.pth'

if not os.path.exists(checkpoint_path):
    print('Downloading SegViT-Base checkpoint (ADE20K, 51.3 mIoU) ...')
    hf_hub_download(
        repo_id='Akide/SegViTv1',
        filename='ade_51.3.pth',
        local_dir='/content/checkpoints',
    )

print(f'Checkpoint ready — {os.path.getsize(checkpoint_path) / 1024**2:.0f} MB')

In [ ]:
import glob

print('Available SegViT configs:')
for p in sorted(glob.glob('configs/segvit/*.py')):
    print(' ', p)

In [ ]:
from mmseg.apis import init_model

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Running on: {device}')

# Update this path if the config listing above shows a different filename
config_file     = 'configs/segvit/segvit_vit-b_jax_640x640_160k_ade20k.py'
checkpoint_file = '/content/checkpoints/ade_51.3.pth'

model = init_model(config_file, checkpoint_file, device=device)
print('Model loaded!')

In [ ]:
from mmseg.apis import inference_model

# Download a sample indoor scene — replace img_path with your own image if preferred
img_path = '/content/sample.jpg'
if not os.path.exists(img_path):
    !wget -q -O /content/sample.jpg \
        'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Bikeroom.jpg/800px-Bikeroom.jpg'

result  = inference_model(model, img_path)

# mmseg 1.x returns SegDataSample; the class-index map is in pred_sem_seg
seg_map = result.pred_sem_seg.data.squeeze(0).cpu().numpy()

print(f'Segmentation map shape:  {seg_map.shape}')
print(f'Unique class indices:    {sorted(set(seg_map.flatten().tolist()))}')

In [ ]:
import matplotlib.pyplot as plt

palette = np.array(model.dataset_meta['palette'], dtype=np.uint8)
classes = model.dataset_meta['classes']

color_seg = palette[seg_map]                                    # (H, W, 3)
img_rgb   = np.array(_PILImage.open(img_path).convert('RGB'))
blend     = (img_rgb * 0.5 + color_seg * 0.5).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_rgb);  axes[0].set_title('Input');         axes[0].axis('off')
axes[1].imshow(blend);    axes[1].set_title('Segmentation');  axes[1].axis('off')
plt.tight_layout()
plt.show()

detected = sorted(set(seg_map.flatten().tolist()))
print('Detected classes:', [classes[i] for i in detected])